# LangChain 기초

**학습 목표**

> 1. 이번 실습에서는 LLM 앱을 만들 때 가장 먼저 필요한 LangChain 기본 구성요소를 익힌다.
> 2. `init_chat_model`을 사용해 모델 식별자만 변경하여 OpenAI, Gemini, Ollama 간 전환을 수행한다.
> 3. **Messages → Prompt Template → Output Parser** 의 표준 처리 흐름을 익힌다.
> 4. **LLM을 호출하고, 프롬프트를 템플릿화하고, 결과를 원하는 형식으로 받는 법**을 익힌다.


---

> **LangChain v1.0의 변경점**
> - `langchain` 네임스페이스가 5개로 단순화됨: `langchain.messages`, `langchain.tools`, `langchain.agents`, `langchain.chat_models`, `langchain.embeddings`.
> - `LLMChain`, 전통 Retriever 등 레거시 기능은 `langchain-classic` 패키지로 분리됨.
> - 본 실습은 v1.0 이상의 API만 사용합니다.
---

# 1. 환경 준비


## (1) 라이브러리 설치

처음 실행하는 환경이라면 아래 셀의 주석을 해제하고 실행합니다. 이미 `pyproject.toml` 또는 `requirements.txt`로 설치했다면 실행하지 않아도 됩니다.

In [1]:
# 필요한 라이브러리 설치
%pip install -U langchain langchain-core langchain-openai langchain-google-genai langchain-groq langchain-ollama python-dotenv pydantic pandas

Note: you may need to restart the kernel to use updated packages.


## (2) 라이브러리 Import

이번 실습에서 사용하는 핵심 객체는 다음과 같습니다.

| 객체 | 역할 |
|---|---|
| `init_chat_model` | `provider:model` 문자열로 여러 제공자의 채팅 모델을 통합 초기화 |
| `PromptTemplate` | 문자열 기반 프롬프트 템플릿 |
| `ChatPromptTemplate` | system/human 메시지를 분리하는 채팅 프롬프트 |
| `StrOutputParser` | 모델 응답에서 문자열만 추출 |


In [2]:
import os
from pathlib import Path
from getpass import getpass
from typing import Literal

import pandas as pd
from dotenv import load_dotenv
from pydantic import BaseModel, Field

from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

## (3) API Key 설정

API 키는 코드에 직접 작성하지 않습니다. 권장 방식은 `.env` 파일에 저장하는 것입니다.

```text
# OpenAI를 사용할 때
OPENAI_API_KEY=sk-...

# Gemini를 사용할 때
GOOGLE_API_KEY=...

```


In [3]:
load_dotenv()

print("OPENAI_API_KEY:", "있음" if os.getenv("OPENAI_API_KEY") else "없음")
print("GOOGLE_API_KEY:", "있음" if os.getenv("GOOGLE_API_KEY") else "없음")

OPENAI_API_KEY: 있음
GOOGLE_API_KEY: 있음


# 2. LangChain이 필요한 이유

OpenAI SDK나 다른 LLM API만으로도 모델 호출은 가능합니다. 예를 들어 "랭체인이 뭐야?"라고 바로 물어볼 수 있습니다.

하지만 실제 서비스나 업무 자동화에서는 단순 호출만으로 충분하지 않습니다.

| 필요 기능 | 실제 상황 예시 |
|---|---|
| 프롬프트 재사용 | 주제만 바꿔 같은 형식의 설명 생성 |
| 역할 부여 | 강사, 면접관, 상담사, 분석가 역할 지정 |
| 출력 형식 고정 | JSON, 표, 리스트, Pydantic 객체 |
| 여러 단계 연결 | 요약 -> 번역 -> 퀴즈 생성 |
| 대량 처리 | 고객 후기 100개를 같은 방식으로 분석 |
| 유지보수 | 프롬프트, 모델, 파서를 분리해 관리 |

LangChain은 LLM 호출을 **구조화된 실행 블록**으로 만들고, 이 블록들을 연결해 앱의 흐름을 구성하는 도구입니다.

# 3. Model

## (1) Model과 `init_chat_model`

- Model은 실제 답변을 생성하는 엔진
- LangChain에서는 모델을 공통 인터페이스로 감싸기 때문에, 이후 프롬프트나 파서와 쉽게 연결할 수 있음

- LangChain v1.x에서는 `init_chat_model("provider:model")` 한 함수로 여러 제공자의 채팅 모델을 통합 초기화할 수 있음.
    - 제공자별 import를 줄일 수 있습니다.
    - 모델 문자열만 바꾸면 OpenAI, Gemini, Groq, Ollama 등으로 전환할 수 있습니다.
    - 이후 `invoke`, `stream`, `batch`, Prompt 연결 방식은 동일하게 유지됩니다.


`CHAT_MODEL`은 `provider:model` 형식을 권장합니다.

| 제공자 | 예시 |
|---|---|
| OpenAI | `openai:gpt-4.1-mini` |
| Gemini | `google_genai:gemini-2.5-flash-lite` |
| Ollama | `ollama:gemma4:e4b` |

### [참고] GPT-5 시리즈는 temperature 고정

GPT-5 / 5.4 / 5.5 모델은 내부에서 multi-pass reasoning 을 수행하기 때문에 답의 무작위성을 막는 `temperature`·`top_p`·`logprob` 파라미터가 사실상 사용 불가 (temperature 는 1 로 고정). `init_chat_model("openai:gpt-5.4-mini")` 처럼 추가 인자 없이 부르면 됩니다.

답의 다양성·톤 조정은 **system 프롬프트**와 **few-shot 예시**로 합니다.

In [4]:
# CHAT_MODEL = 'openai:gpt-4.1-mini'
# CHAT_MODEL = 'google_genai:gemini-2.5-flash-lite'
CHAT_MODEL = 'openai:gpt-4.1-mini'
model = init_chat_model(CHAT_MODEL)
model

ChatOpenAI(output_version=None, profile={'name': 'GPT-4.1 mini', 'release_date': '2025-04-14', 'last_updated': '2025-04-14', 'open_weights': False, 'max_input_tokens': 1047576, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x000001C23A9E06E0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001C23A9E1160>, root_client=<openai.OpenAI object at 0x000001C239FEBB60>, root_async_client=<openai.AsyncOpenAI object at 0x000001C23A9E0EC0>, model_name='gpt-4.1-mini', model_kwargs={

- `init_chat_model`로 만든 모델도 다른 채팅 모델과 동일하게 `invoke()`로 호출합니다. 반환값은 메시지 객체이며, 실제 답변 텍스트는 `.content`에서 확인합니다.

In [5]:
response = model.invoke('좀비가 카페를 운영한다면 첫날 메뉴판 5개를 짧게 뽑아줘.')
print(response.content)

물론이죠! 좀비가 운영하는 카페 첫날 메뉴판, 짧고 간단하게 뽑아봤어요:

1. 뇌맛 라떼  
2. 피의 아메리카노  
3. 썩은 과일 스무디  
4. 좀비 브레드 토스트  
5. 핏빛 레드벨벳 케이크  

필요하면 더 재미있게도 만들어드릴 수 있어요!


## (2) `ChatOpenAI` 사용

In [6]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model='gpt-4.1-mini',   # 사용할 OpenAI 모델 이름
    timeout=30,             # 응답 대기 시간 제한 : 30초
    max_retries=3,          # 요청 실패 시 최대 3번까지 재시도
)

model

ChatOpenAI(output_version=None, profile={'name': 'GPT-4.1 mini', 'release_date': '2025-04-14', 'last_updated': '2025-04-14', 'open_weights': False, 'max_input_tokens': 1047576, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x000001C23AA90550>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001C23AA90F50>, root_client=<openai.OpenAI object at 0x000001C23AA902D0>, root_async_client=<openai.AsyncOpenAI object at 0x000001C23AA90CD0>, model_name='gpt-4.1-mini', model_kwargs={

- `invoke()`는 하나의 입력을 넣고 하나의 출력을 받는 가장 기본적인 실행 방식입니다.

In [7]:
response = model.invoke("LangChain을 처음 배우는 사람에게 아주 친절하게 한번 봐도 이해가 될 정도로 한 문단으로 설명해줘")
print(response.content)

LangChain은 여러 가지 언어 모델(예: GPT)을 쉽고 효율적으로 연결해서 복잡한 작업을 자동화할 수 있게 도와주는 도구예요. 예를 들어, 질문에 답하거나, 문서를 요약하거나, 새로운 글을 쓸 때 단순히 모델에 입력을 넣는 것뿐 아니라, 여러 단계의 작업을 차례로 처리하거나 외부 데이터베이스와 연동해서 더 똑똑한 결과를 만들 수 있게 해주죠. 그래서 코딩 초보자도 미리 만들어진 함수와 템플릿을 활용해 AI 기능을 손쉽게 프로젝트에 적용할 수 있도록 설계된 프레임워크라고 생각하면 돼요.


- 응답 객체에는 답변 본문뿐 아니라 모델명, 토큰 사용량 같은 메타데이터가 포함될 수 있습니다. 
- 실제 서비스에서는 사용량 추적이나 로깅에 활용할 수 있습니다.

In [8]:
print("content:", response.content)
print("response_metadata:", response.response_metadata)
print("usage_metadata:", response.usage_metadata)

content: LangChain은 여러 가지 언어 모델(예: GPT)을 쉽고 효율적으로 연결해서 복잡한 작업을 자동화할 수 있게 도와주는 도구예요. 예를 들어, 질문에 답하거나, 문서를 요약하거나, 새로운 글을 쓸 때 단순히 모델에 입력을 넣는 것뿐 아니라, 여러 단계의 작업을 차례로 처리하거나 외부 데이터베이스와 연동해서 더 똑똑한 결과를 만들 수 있게 해주죠. 그래서 코딩 초보자도 미리 만들어진 함수와 템플릿을 활용해 AI 기능을 손쉽게 프로젝트에 적용할 수 있도록 설계된 프레임워크라고 생각하면 돼요.
response_metadata: {'token_usage': {'completion_tokens': 152, 'prompt_tokens': 35, 'total_tokens': 187, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_4f7ecf0cc2', 'id': 'chatcmpl-Dp7rGiCSU9AlPJPnc1hxZjVpZslH6', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}
usage_metadata: {'input_tokens': 35, 'output_tokens': 152, 'total_tokens': 187, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reaso

## (3) Temperature 비교

- `temperature`는 답변의 다양성과 예측 가능성을 조절하는 값

| 값 | 특징 |
|---|---|
| 0에 가까움 | 안정적, 반복 실행 시 비슷한 결과 |
| 1에 가까움 | 다양하고 창의적인 결과 |


In [21]:
question = '생성형 ai를 중학생에게 설명해줘.'

cold_model = init_chat_model(CHAT_MODEL, temperature=0)
creative_model = init_chat_model(CHAT_MODEL, temperature=1.0)

In [22]:

print("[temperature=0]")
print(cold_model.invoke(question).content)

print("\n[temperature=1]")
print(creative_model.invoke(question).content)

[temperature=0]
물론이지! 생성형 AI를 중학생도 쉽게 이해할 수 있게 설명해볼게.

생성형 AI는 '새로운 것'을 만들어 내는 인공지능이야. 예를 들어, 그림을 그리거나, 글을 쓰거나, 음악을 만드는 것처럼 사람이 하는 창작 활동을 컴퓨터가 할 수 있게 도와주는 기술이야.

이 AI는 많은 데이터를 공부해서 배워. 예를 들어, 수천 개의 그림을 보고 어떤 그림이 어떻게 생겼는지 배우고, 그걸 바탕으로 새로운 그림을 그릴 수 있어. 또는 많은 글을 읽고 나서 새로운 이야기를 만들어 낼 수도 있지.

쉽게 말하면, 생성형 AI는 '배운 것을 바탕으로 새로운 것을 만들어 내는 똑똑한 컴퓨터 친구'라고 생각하면 돼!

[temperature=1]
생성형 AI는 '새로운 것'을 만들어 내는 인공지능이에요. 예를 들어, 그림을 그리거나, 이야기를 쓰거나, 음악을 만드는 것처럼 사람이 직접 하지 않아도 컴퓨터가 스스로 멋진 결과물을 만들어 내는 거죠.

중학생인 네가 생각해볼 수 있는 예로, 만약 너가 ‘강아지가 공원에서 놀고 있는 그림’을 그려달라고 하면, 생성형 AI는 그런 그림을 만들어 줄 수 있어요. 또, 짧은 이야기나 시를 만들어 주기도 하고, 질문에 답을 하거나 대화도 할 수 있답니다.

쉽게 말해서, 생성형 AI는 배우고 연습해서 새로운 아이디어나 작품을 스스로 ‘창작’할 수 있는 똑똑한 컴퓨터 친구라고 생각하면 돼요!


## [실습] 모델 호출 바꿔보기

모델을 gemini 등 다른 모델로 바꿔봅니다.
그리고 아래 질문을 바꿔 실행해 봅니다.

- "RAG를 비전공자에게 설명해줘."
- "AI Agent를 회사 업무 자동화 예시로 설명해줘."
- "LangGraph와 LangChain의 차이를 간단히 설명해줘."

In [23]:
question2 = 'RAG를 비전공자에게 설명해줘.'


print("[temperature=0]")
print(cold_model.invoke(question2).content)

print("\n[temperature=1]")
print(creative_model.invoke(question2).content)

[temperature=0]
물론이죠! RAG에 대해 비전공자도 이해하기 쉽게 설명해드릴게요.

---

**RAG란?**

RAG는 "Retrieval-Augmented Generation"의 약자예요. 쉽게 말해서, 컴퓨터가 뭔가를 대답할 때 **기억 속에 있는 정보뿐만 아니라, 필요한 정보를 인터넷이나 데이터베이스에서 찾아서** 더 정확하고 풍부하게 답변하는 기술이에요.

---

**좀 더 쉽게 풀어볼게요:**

- 우리가 모르는 질문을 받았을 때, 그냥 머릿속 기억만으로 답하면 틀릴 수도 있잖아요?
- 그래서 책이나 인터넷에서 필요한 정보를 찾아보고, 그걸 바탕으로 대답하면 더 정확하겠죠?
- RAG는 컴퓨터가 이런 식으로 작동하게 만드는 기술이에요.

---

**예를 들어볼까요?**

- 당신이 "2023년에 노벨 평화상은 누가 받았어?"라고 물어봤어요.
- 컴퓨터가 그냥 자기 기억(학습된 데이터)만으로 답하면 틀릴 수도 있어요.
- 하지만 RAG 시스템은 최신 정보를 인터넷이나 데이터베이스에서 찾아서, "2023년 노벨 평화상은 누구누구가 받았어요"라고 정확히 알려줄 수 있어요.

---

**왜 중요할까요?**

- 컴퓨터가 더 똑똑해지고, 최신 정보도 반영할 수 있어요.
- 단순히 미리 배운 내용만 말하는 게 아니라, 필요한 정보를 직접 찾아서 알려주니까 답변이 훨씬 신뢰할 만해져요.

---

요약하자면,  
**RAG는 컴퓨터가 '기억 + 검색'을 합쳐서 더 똑똑하게 대답하도록 돕는 기술**이라고 생각하면 됩니다!

궁금한 점 있으면 언제든 물어보세요!

[temperature=1]
물론이에요! RAG에 대해 비전공자도 쉽게 이해할 수 있도록 설명해볼게요.

---

**RAG란?**

RAG는 "Retrieval-Augmented Generation"의 약자예요. 영어 뜻을 쉽게 풀어보면 "정보를 찾아서 그것을 바탕으로 글을 만들어 내는 기술"이라고 할 수 있어요.

---

**왜 필요한가요?**

예를 들어, 우리가 어떤 질문을 했을

# 4. Prompt

- Prompt는 LLM에게 주는 작업 지시서
- 좋은 프롬프트는 단순히 질문을 잘 쓰는 것이 아니라, 다음 요소를 설계하는 일

| 요소 | 예시 |
|---|---|
| 역할 | "너는 AI 강의를 하는 친절한 강사야." |
| 목적 | "초보자가 이해할 수 있게 설명해줘." |
| 제약 | "5문장 이내로 작성해줘." |
| 출력 형식 | "표로 정리해줘.", "JSON으로 답해줘." |
| 예시 | "아래 예시와 같은 톤으로 작성해줘." |

LangChain에서는 반복되는 프롬프트를 템플릿으로 만들고, 변수만 바꿔 재사용합니다.

## (1) PromptTemplate

- `PromptTemplate`은 문자열 기반 템플릿
- `{topic}` 같은 변수를 넣고, 실행할 때 실제 값을 전달

In [14]:
basic_prompt = PromptTemplate.from_template(
    '''
다음 개념을 초등학생도 이해할 수 있게 설명해줘.

개념 : {topic}

조건 :
- 어려운 용어는 풀어서설명한다.
- 일상생활 비유를 1개 포함한다.
- 마지막에 한 줄 요약을 작성한다.
'''.strip()
)

formatted_prompt = basic_prompt.invoke({"topic":"Self-Attention"})

# 아직 LLM 호출한 것은 아니고
print(formatted_prompt.text)

다음 개념을 초등학생도 이해할 수 있게 설명해줘.

개념 : Self-Attention

조건 :
- 어려운 용어는 풀어서설명한다.
- 일상생활 비유를 1개 포함한다.
- 마지막에 한 줄 요약을 작성한다.


## (2) ChatPromptTemplate

- 채팅 모델에는 `ChatPromptTemplate`이 더 자주 사용됨
- 시스템 메시지, 사용자 메시지, AI 메시지 등 역할(role) 구분
- 다중 메시지 기반의 프롬프트 흐름을 구성할 수 있도록 도와주는 템플릿

| 메시지 역할 | 의미 |
|---|---|
| `System` | 모델의 역할, 원칙, 톤. AI에게 역할/성격을 지정 |
| `Human` | 실제 사용자 질문 또는 요청 |
| `AI` | AI 응답 |

In [16]:
chat_prompt = ChatPromptTemplate.from_messages([
('system', '너는 생성형 AI와 Langchain에 대해 쉽게 설명을 해주는 멘토다. 한국어로 대답해줘'),
('human', '주제: {topic}\n 대상: {audience}\n 요청: 쉬운 설명, 예시 2개, 확인 질문 1개를 작성해줘.')
])

messages = chat_prompt.invoke({
    'topic':'프롬프트 엔지니어링',
    'audience':'LLM을 처음 배우는 비전공자'
})

messages

ChatPromptValue(messages=[SystemMessage(content='너는 생성형 AI와 Langchain에 대해 쉽게 설명을 해주는 멘토다. 한국어로 대답해줘', additional_kwargs={}, response_metadata={}), HumanMessage(content='주제: 프롬프트 엔지니어링\n 대상: LLM을 처음 배우는 비전공자\n 요청: 쉬운 설명, 예시 2개, 확인 질문 1개를 작성해줘.', additional_kwargs={}, response_metadata={})])

In [17]:
result = model.invoke(messages)
print(result.content)

안녕하세요! 오늘은 **프롬프트 엔지니어링**에 대해 쉽게 설명해드릴게요.

---

### 프롬프트 엔지니어링이란?

프롬프트 엔지니어링은 AI에게 원하는 답을 얻기 위해 **질문(프롬프트)**을 잘 만드는 기술이에요. AI는 우리가 입력한 문장을 보고 답을 하기 때문에, 질문을 어떻게 만드느냐가 결과에 큰 영향을 줍니다.

쉽게 말해, AI가 똑똑하게 답할 수 있도록 ‘길잡이 문장’을 만드는 것이죠!

---

### 예시 1: 간단한 질문

- 안 좋은 프롬프트:  
  “영화 알려줘”

- 좋은 프롬프트:  
  “2020년 이후 개봉한 재미있는 코미디 영화 3편 추천해줘.”

  
**설명**: 두 번째 문장이 더 구체적이고 명확해서 AI가 더 정확한 답을 할 수 있어요.

---

### 예시 2: 역할을 지정하는 문장 추가

- 안 좋은 프롬프트:  
  “날씨 알려줘”

- 좋은 프롬프트:  
  “기상 전문가처럼 행동해서 오늘 서울의 날씨를 알려줘.”

  
**설명**: AI가 마치 전문가처럼 답하게끔 상황을 설정해줘서 더 신뢰성 있는 답변을 받을 수 있습니다.

---

### 확인 질문

“AI에게 질문할 때, 왜 구체적이고 명확한 프롬프트가 중요할까요?”

---

필요하면 더 궁금한 점도 물어봐 주세요!


## [실습] 역할 프롬프트 만들기

`system` 역할을 바꿔 같은 주제의 답변이 어떻게 달라지는지 확인합니다.

예시 역할:

- "너는 초등학생에게 설명하는 과학 선생님이다."
- "너는 기업 임원에게 보고하는 AI 컨설턴트다."
- "너는 개발자에게 코드 중심으로 설명하는 시니어 엔지니어다."

In [24]:
chat_prompt = ChatPromptTemplate.from_messages([
('system', '너는 초등학생에게 설명하는 과학 선생님이다. 한국어로 대답해줘'),
('human', '주제: {topic}\n 대상: {audience}\n 요청: 쉬운 설명, 예시 2개, 확인 질문 1개를 작성해줘.')
])

message2 = chat_prompt.invoke({
    'topic':'프롬프트 엔지니어링',
    'audience':'LLM을 처음 배우는 초등학생'
})

result = model.invoke(message2)
print(result.content)

안녕! 오늘은 ‘프롬프트 엔지니어링’에 대해 쉽게 설명해 줄게.

**프롬프트 엔지니어링이란 무엇일까?**  
프롬프트는 “말을 걸 때 쓰는 문장”이라고 생각하면 돼. 예를 들어, 친구한테 “오늘 뭐 할까?”라고 물어보는 것처럼 컴퓨터한테도 우리가 원하는 답을 얻으려면 잘 물어봐야 해. 프롬프트 엔지니어링은 컴퓨터, 특히 똑똑한 언어 프로그램(LLM)에게 잘 물어보는 방법을 만드는 거야.

**쉬운 예시 2가지!**  
1. **예시 1:**  
만약 컴퓨터에게 “동물에 대해 알려줘”라고 물어보면 아주 긴 답이 나올 수 있어.  
그래서 이렇게 말해보자!  
“강아지에 대해 재미있고 간단하게 말해줘.”  
이렇게 하면 딱 원하는 답을 얻을 수 있어!

2. **예시 2:**  
“세계에서 제일 높은 산이 뭐야?”  
라고 물어보는 대신에,  
“세계에서 제일 높은 산에 대해 학교 친구에게 설명하듯 쉽게 알려줘.”  
라고 말하면 더 친절하고 알기 쉬운 답이 나올 거야.

**확인 질문!**  
컴퓨터에게 원하는 답을 얻으려면 어떻게 질문해야 할까?  
(힌트: 구체적이고 친절하게 물어보는 거야!)

잘 이해했니? 궁금한 거 있으면 언제든 물어봐!


## [실습] 영화 추천 템플릿 만들기
- 입력변수 : 장르
- 장르를 입력받아 영화 1편과 추천이유를 설명하는 템플릿을 만들고 사용해 봅시다.

In [25]:
chat_prompt = ChatPromptTemplate.from_messages([
('system', '너는 시청자에게 설명하는 영화 평론가이다. 한국어로 대답해줘'),
('human', '주제: {topic}\n 대상: {audience}\n 장르: {genre}\n 요청: 쉬운 설명, 추천 영화 1개, 확인 질문 1개를 작성해줘.')
])

message3 = chat_prompt.invoke({
    'topic':'영화',
    'audience':'영화를 좋아하는 남자 중학생',
    'genre':'누아르'
})

result = model.invoke(message3)
print(result.content)

안녕! 오늘은 누아르 영화에 대해 쉽게 설명해줄게. 누아르는 보통 어두운 분위기와 복잡한 이야기, 그리고 주인공들이 도덕적으로 모호한 모습을 가진 영화 장르야. 주인공들이 때로는 잘못된 선택을 하거나 위험한 상황에 빠지면서 긴장감이 커지는 게 특징이지.

누아르 영화 추천작으로는 ‘블레이드 러너’라는 영화를 추천할게. 이 영화는 미래 도시를 배경으로 어둡고 신비로운 분위기를 잘 보여주고, 주인공이 어떤 진실을 찾아가는 이야기가 매우 흥미로워.

혹시 너는 평소에 어떤 스타일의 영화나 이야기를 좋아하니? 궁금해!


# 5. Output Parser

- Output Parser는 LLM에서 반환된 자유형 텍스트(string)를 우리가 원하는 형태로 가공해주는 도구

- LLM의 응답은 기본적으로 메시지 객체이고, 사람이 읽을 때는 `response.content`만 확인하면 되지만, 프로그램에서는 결과를 일정한 형태로 다루는 것이 중요함


| Parser | 역할 | 사용 상황 |
|---|---|---|
| `StrOutputParser` | 응답에서 문자열만 추출 | 일반 답변, 이메일, 요약 |
| JSON 계열 Parser | JSON 문자열을 dict로 변환 | API 응답처럼 쓰고 싶을 때 |
| Pydantic 구조화 출력 | 스키마에 맞는 객체로 검증 | 분류, 추출, 업무 자동화 |

최근 LangChain에서는 모델의 `with_structured_output()` 기능을 사용해 Pydantic 모델로 결과를 받는 방식도 많이 사용합니다.

## (1) StrOutputParser

`StrOutputParser`를 체인 끝에 붙이면 모델 응답 객체에서 텍스트만 꺼내 줍니다.

## (2) PydanticOutputParser

#### 1) Pydantic
- 파이썬에서 데이터 형태를 정의하고 검증하는 라이브러리

In [11]:
class User(BaseModel):
    name: str
    age: int

#### 2) 출력파서로 이용
- llm의 성능에 따라 출력 파싱에 맞게 적절한  답변을 할 수도 있고, 잘못 답변해서 오류가 날 수도 있음.

## (3) 구조화 출력

업무 자동화에서는 자유 텍스트보다 정해진 구조가 더 유용할 때가 많습니다.

예를 들어 고객 후기를 분석한다면 다음처럼 감성, 카테고리, 우선순위, 다음 조치를 분리해서 받아야 이후 시스템에서 활용하기 쉽습니다.

In [12]:
class CustomerIssue(BaseModel):
    sentiment: 

    category: 
    priority: 
    summary: 
    next_action: 


issue_model =

review = 
issue = 

issue

SyntaxError: invalid syntax (1095887262.py, line 2)

## [실습] 감성 분석 결과 구조화

아래 리뷰를 분석해 다음 필드를 가진 `MovieReview` 모델을 만들어 봅니다.

입력 문장:

```text
이 영화는 영상미는 좋았지만 스토리가 너무 지루했다.
```

출력 목표:

```json
{
  "sentiment": "mixed",
  "positive": "영상미가 좋음",
  "negative": "스토리가 지루함",
  "recommendation": "시각적 연출을 좋아하는 관객에게만 추천"
}
```

## [실습] 게임 캐릭터 카드
- 게임 캐릭터 이름, 직업, 성격, 대표 특기 또는 필살기, 약점를 나타내는 게임 캐릭터 카드를 만들어보세요.